# Phase PGVIB — Multi-seed Replication + xlsr-300m g0-only (Runtime 3)

Two phases queued back-to-back on Colab Runtime 3, sharing one env-setup pass.

## Phase 1 — Multi-seed replication (per-group VIB + drop-g0)

**Goal:** turn the existing seed-42 per-group VIB and drop_vib_g0 single-point results into 3-seed mean ± std measurements. Seed 42 is already done (per-group: 11.67%, drop-g0: 17.61%). This phase adds seeds 123 and 456 for both.

The contribution claim — that monolithic VIB is suboptimal and g0 carries the bottleneck signal — depends on the per-group ↔ drop-g0 gap being stable across seeds. A 6-point gap at one seed is suggestive; the same gap with mean ± std across three seeds is evidence.

**Variants (4 runs):**
1. `pergroup_vib_s123` — full per-group VIB, seed 123
2. `pergroup_vib_s456` — full per-group VIB, seed 456
3. `drop_vib_g0_s123` — VIB disabled on g0, seed 123
4. `drop_vib_g0_s456` — VIB disabled on g0, seed 456

All other knobs match the existing seed-42 reference.

## Phase 2 — Cross-backbone (xlsr-300m g0-only, 1 run)

**Goal:** show the g0-only finding generalizes beyond the WavLM family. xlsr-300m is a wav2vec2-lineage multilingual SSL model — distinct training objective and data from WavLM. 4 groups, so `vib_disabled_groups=(1, 2, 3)` for g0-only. batch_size=24 to fit on A100; grad_accum_steps auto-recomputed.

**Caveat:** single-seed directional check, not apples-to-apples on every knob.

## Combined sweep

Single `VARIANTS` dict with 5 entries. Wall time ≈ 18-20 hr (within Colab Pro 24 hr session limit). Splittable later by name prefix (`pergroup_vib_` / `drop_vib_g0_` / `g0_only_xlsr_`).

Self-contained: Run All from scratch.


## 1. Environment Setup


In [2]:
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    raise RuntimeError('No GPU detected')


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB (85.1 GB)


## 2. Copy GLADA Package to Local Runtime


In [3]:
import shutil, os, sys

CANDIDATES = [
    "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code/glada",
    '/content/drive/MyDrive/Jakob/glada_pkg/glada',
    '/content/drive/MyDrive/Jakob/glada_pkg',
    '/content/drive/MyDrive/Jakob/glada',
]
DRIVE_PKG = next((c for c in CANDIDATES if os.path.exists(os.path.join(c, '__init__.py'))), None)
if DRIVE_PKG is None:
    raise FileNotFoundError(
        'No glada/__init__.py found in candidates. Run:\n'
        "  !find /content/drive -name '__init__.py' -path '*/glada/*' 2>/dev/null | head\n"
        'and add the parent directory to CANDIDATES above.'
    )

LOCAL_PKG = '/content/glada'
if os.path.exists(LOCAL_PKG): shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)

for mod in list(sys.modules):
    if mod == 'glada' or mod.startswith('glada.'): del sys.modules[mod]
sys.path = [p for p in sys.path if 'glada' not in p]
if '/content' not in sys.path: sys.path.insert(0, '/content')
print(f'Package source: {DRIVE_PKG}\nCopied to:      {LOCAL_PKG}')


Package source: /content/drive/MyDrive/Jakob/glada_pkg/glada
Copied to:      /content/glada


## 3. Datasets (ASVspoof 2019 LA + ITW)


In [4]:
import tarfile

def extract_dataset(tar_path, extract_to, name):
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f'{name}: already extracted to {extract_to}'); return
    os.makedirs(extract_to, exist_ok=True)
    print(f'{name}: extracting {tar_path} -> {extract_to} ...')
    with tarfile.open(tar_path) as t: t.extractall(extract_to)
    print(f'{name}: done.')

extract_dataset('/content/drive/MyDrive/Jakob/Datasets/LA.tar', '/content/data/asvspoof', 'ASVspoof 2019 LA')
extract_dataset('/content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar', '/content/data/itw', 'In-the-Wild')

ASV_PROTO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_AUDIO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
ITW_META       = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO      = '/content/data/itw/release_in_the_wild'
for p in [ASV_PROTO, ASV_AUDIO, ASV_EVAL_PROTO, ITW_META]:
    assert os.path.exists(p), f'Missing: {p}'
print('Datasets verified.')


ASVspoof 2019 LA: extracting /content/drive/MyDrive/Jakob/Datasets/LA.tar -> /content/data/asvspoof ...


/tmp/ipykernel_424/3241063313.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tar_path) as t: t.extractall(extract_to)


ASVspoof 2019 LA: done.
In-the-Wild: extracting /content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar -> /content/data/itw ...
In-the-Wild: done.
Datasets verified.


## 4. Imports


In [5]:
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.datasets import (
    ASVspoofDatasetWithSpeaker,
    ASVspoofEvalDataset,
    InTheWildDataset,
    create_dataloaders,
)
from glada.augmentation import RawBoost
from glada.models import GLADAModel
from glada.models.vib import VIBBottleneck
from glada.models.grl import GradientReversal
from glada.models.discriminators import GroupedLayerDomainDiscriminators
from glada.trainers import GLADATrainer

import copy, gc, json, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

print('Imports OK')


Imports OK


## 5. INLINE: `PerGroupVIBModel` (with leave-one-out + per-group beta flags)

Subclasses `GLADAModel`. Adds K `VIBBottleneck` modules (one per group), each compressing a `(B, 2D)` group ASP output to `(B, bottleneck_dim)`. The per-group discriminator then receives the compressed `z_k` rather than the raw `g_k`.

**New flags vs. the phase_vib_stagger version:**
- `vib_disabled_groups: tuple = ()` — for these group indices, `z_k = mu_k` (deterministic, no reparam sampling) and `kl_k = 0`. Used by `phase_a_leaveout` to ablate one group's compression at a time.
- `vib_betas: tuple = None` — per-group beta override. If provided, `kl_total = sum_k vib_betas[k] * kl_k` instead of the default uniform mean. Used by `phase_b_confirm` to apply scaled betas based on Phase A leave-one-out results.

Backwards compatibility: with no extra kwargs, behavior matches the original phase_vib_stagger inline class.


In [6]:
class PerGroupVIBModel(GLADAModel):
    """GLADAModel with one VIBBottleneck per group, applied BEFORE the
    per-group discriminators. The hierarchical pool + global VIB are
    replaced by a simple linear projection over the concatenated
    per-group compressed representations.

    Args (in addition to GLADAConfig):
        vib_disabled_groups: tuple of group indices to disable VIB on
            (z_k = mu_k, kl_k = 0). Used for leave-one-out ablation.
        vib_betas: tuple of length K with per-group beta multipliers.
            None -> uniform mean (original behavior). Tuple -> sum of
            beta-weighted KLs.
    """

    def __init__(self, config, vib_disabled_groups=(), vib_betas=None):
        super().__init__(config)
        K = config.num_groups
        D = config.hidden_dim
        bd = config.bottleneck_dim
        self.group_vibs = nn.ModuleList([
            VIBBottleneck(2 * D, bd) for _ in range(K)
        ])
        self.group_compressed_proj = nn.Linear(K * bd, bd)
        if self.glada_disc is not None:
            self.glada_disc = GroupedLayerDomainDiscriminators(
                num_groups=K,
                input_dim=bd,
                hidden_dim=config.glada_disc_hidden,
            )

        # Phase A controls.
        for k in vib_disabled_groups:
            assert 0 <= k < K, f'vib_disabled_groups index {k} not in [0,{K})'
        self._vib_disabled = set(vib_disabled_groups)
        if vib_betas is not None:
            assert len(vib_betas) == K, f'vib_betas must have len {K}, got {len(vib_betas)}'
            self.register_buffer('_vib_betas',
                torch.tensor(vib_betas, dtype=torch.float32))
        else:
            self._vib_betas = None

    def forward(self, input_values, attention_mask=None):
        backbone_frozen = not any(p.requires_grad for p in self.backbone.parameters())
        if backbone_frozen:
            with torch.no_grad():
                outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        else:
            outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        hidden_states = list(outputs.hidden_states)

        if attention_mask is not None:
            input_lengths = attention_mask.sum(dim=1)
            output_lengths = ((input_lengths - 1) // 320) + 1
            T = hidden_states[0].shape[1]
            frame_mask = (torch.arange(T, device=attention_mask.device).unsqueeze(0) < output_lengths.unsqueeze(1))
        else:
            frame_mask = None

        hidden_states = self.layer_norms(hidden_states)
        _, cka_loss, group_outputs = self.pooling(hidden_states, frame_mask=frame_mask)
        if not self.config.use_cka_loss:
            cka_loss = torch.tensor(0.0, device=group_outputs[0].device)

        # Per-group VIB with leave-one-out + per-group beta support.
        z_groups = []
        kl_per_group = []
        for k, g_k in enumerate(group_outputs):
            if k in self._vib_disabled:
                # Disabled: deterministic z_k = mu_k, kl_k = 0.
                z_k, _ = self.group_vibs[k](g_k, training=False)
                kl_k = torch.zeros((), device=g_k.device)
            else:
                z_k, kl_k = self.group_vibs[k](g_k, training=self.training)
            z_groups.append(z_k)
            kl_per_group.append(kl_k)

        # Aggregate KL across groups.
        if self._vib_betas is not None:
            # Beta-weighted sum (per-group beta).
            kl_total = sum(self._vib_betas[k] * kl_per_group[k] for k in range(len(group_outputs)))
        else:
            # Uniform mean (original behavior; effective per-group beta = config.vib_beta / G).
            kl_total = sum(kl_per_group) / max(len(group_outputs), 1)

        z_concat = torch.cat(z_groups, dim=-1)
        z = self.group_compressed_proj(z_concat)

        spoof_logits = self.task_classifier(z)
        result = {
            'spoof_logits': spoof_logits,
            'kl_loss': kl_total,
            'cka_loss': cka_loss,
            'features': z,
        }

        if self.glada_disc is not None:
            result['glada_domain_logits'] = self.glada_disc(z_groups, self.domain_grl_alpha)
            result['group_outputs'] = z_groups

        if self.domain_disc is not None:
            z_domain = GradientReversal.apply(z, self.domain_grl_alpha)
            result['domain_logits'] = self.domain_disc(z_domain)

        if self.speaker_adversary is not None:
            z_speaker = GradientReversal.apply(z, self.speaker_grl_alpha)
            result['speaker_logits'] = self.speaker_adversary(z_speaker)

        return result

print('PerGroupVIBModel defined (with vib_disabled_groups + vib_betas).')


PerGroupVIBModel defined (with vib_disabled_groups + vib_betas).


## 7. Configuration Helper


In [7]:
USE_FULL_EPOCHS = False
PHASE1 = 5  if USE_FULL_EPOCHS else 3
PHASE2 = 45 if USE_FULL_EPOCHS else 17

DRIVE_LOG_DIR = '/content/drive/MyDrive/Jakob/Experiments/phase_pgvib_replicate'
os.makedirs(DRIVE_LOG_DIR, exist_ok=True)

def make_config(name, backbone='wavlm-base-plus', seed=42):
    cfg = GLADAConfig(
        experiment_name=name,
        backbone=backbone,
        seed=seed,
        batch_size=64,  # FIXED for apples-to-apples vs 11.67% reference
        num_workers=0,
        phase1_epochs=PHASE1,
        phase2_epochs=PHASE2,
        drive_log_dir=DRIVE_LOG_DIR,
        run_id=time.strftime('%Y%m%d_%H%M%S'),
    )
    return cfg


## 8. Datasets (RawBoost=ON)


In [8]:
base_for_data = make_config('_data_setup')

source_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=ASV_PROTO,
    audio_dir=ASV_AUDIO,
    config=base_for_data,
    augmenter=RawBoost,  # RawBoost only — matches phase_vib_stagger reference
)
target_ds = InTheWildDataset(
    meta_csv_path=ITW_META, audio_dir=ITW_AUDIO,
    config=base_for_data, use_labels=False,
)
eval_ds = InTheWildDataset(
    meta_csv_path=ITW_META, audio_dir=ITW_AUDIO,
    config=base_for_data, use_labels=True,
)
extra_eval = {
    '2019_LA': ASVspoofEvalDataset(
        protocol_path=ASV_EVAL_PROTO, audio_dir=ASV_EVAL_AUDIO, config=base_for_data,
    ),
}
num_speakers = len(source_ds.speaker_to_id) if hasattr(source_ds, 'speaker_to_id') else None
print(f'Source: {len(source_ds):,} ({num_speakers} speakers)')
print(f'Target: {len(target_ds):,}, Eval: {len(eval_ds):,}, 2019 LA: {len(extra_eval["2019_LA"]):,}')


  ASVspoof: 25380 samples, 20 speakers, rawboost=0.5
  ITW: 31779 samples (unlabeled returned)
  ITW: 31779 samples (labeled returned)
  ASVspoof Eval: 71237 samples
Source: 25,380 (20 speakers)
Target: 31,779, Eval: 31,779, 2019 LA: 71,237


## 8b. Variant Definitions (Phase 1 + Phase 2)

Five runs total. Phase 1 = 4 replication runs on Base-Plus. Phase 2 = 1 cross-backbone run on xlsr-300m (4 groups).


In [9]:
RAWBOOST_DEFAULT = True  # matches existing seed-42 references
RESULTS_FILENAME = 'phase_pgvib_replicate_results.json'

VARIANTS = {
    # === Phase 1: replication (Base-Plus, 2 extra seeds for each condition) ===
    'pergroup_vib_s123': {
        'model_class': 'PerGroupVIBModel',
        'model_kwargs': {},
        'config_overrides': {'seed': 123},
    },
    'pergroup_vib_s456': {
        'model_class': 'PerGroupVIBModel',
        'model_kwargs': {},
        'config_overrides': {'seed': 456},
    },
    'drop_vib_g0_s123': {
        'model_class': 'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (0,)},
        'config_overrides': {'seed': 123},
    },
    'drop_vib_g0_s456': {
        'model_class': 'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (0,)},
        'config_overrides': {'seed': 456},
    },
    # === Phase 2: xlsr-300m g0-only (cross-backbone, 1 run, seed 42) ===
    'g0_only_xlsr_s42': {
        'model_class': 'PerGroupVIBModel',
        'backbone': 'xlsr-300m',
        'model_kwargs': {'vib_disabled_groups': (1, 2, 3)},  # xlsr-300m has 4 groups
        'config_overrides': {'seed': 42, 'batch_size': 24},
    },
}
for name, spec in VARIANTS.items():
    bb = spec.get('backbone', 'wavlm-base-plus (default)')
    print(f'  {name:<24s} backbone={bb:<20s} '
          f'seed={spec["config_overrides"]["seed"]}, kwargs={spec["model_kwargs"]}')


  pergroup_vib_s123        backbone=wavlm-base-plus (default) seed=123, kwargs={}
  pergroup_vib_s456        backbone=wavlm-base-plus (default) seed=456, kwargs={}
  drop_vib_g0_s123         backbone=wavlm-base-plus (default) seed=123, kwargs={'vib_disabled_groups': (0,)}
  drop_vib_g0_s456         backbone=wavlm-base-plus (default) seed=456, kwargs={'vib_disabled_groups': (0,)}
  g0_only_xlsr_s42         backbone=xlsr-300m            seed=42, kwargs={'vib_disabled_groups': (1, 2, 3)}


## 8c. Sanity Probe

One-batch source-loader probe before launching 12 hours of training.


In [10]:
print('Sanity-checking source loader (3 batches)...')
sanity_loader = torch.utils.data.DataLoader(source_ds, batch_size=4, shuffle=True, num_workers=0)
for i, batch in enumerate(sanity_loader):
    if i >= 3: break
    spk = batch.get('speaker_id')
    lbl = batch.get('task_label')
    assert lbl is not None, f'No task_label in batch keys: {list(batch.keys())}'
    assert lbl.min() >= 0 and lbl.max() <= 1, f'task_label out of range: [{int(lbl.min())}, {int(lbl.max())}]'
    if spk is not None:
        assert spk.min() >= 0 and spk.max() < num_speakers, \
            f'speaker_id out of range: [{int(spk.min())}, {int(spk.max())}] vs num_speakers={num_speakers}'
    shape_summary = [(k, tuple(v.shape)) for k, v in batch.items() if hasattr(v, 'shape')]
    print(f'  batch {i}: shapes={shape_summary}, task_labels={lbl.unique().tolist()}')
print('Source loader sanity OK.')


Sanity-checking source loader (3 batches)...
  batch 0: shapes=[('input_values', (4, 64000)), ('attention_mask', (4, 64000)), ('task_label', (4,)), ('speaker_id', (4,))], task_labels=[0]
  batch 1: shapes=[('input_values', (4, 64000)), ('attention_mask', (4, 64000)), ('task_label', (4,)), ('speaker_id', (4,))], task_labels=[0, 1]
  batch 2: shapes=[('input_values', (4, 64000)), ('attention_mask', (4, 64000)), ('task_label', (4,)), ('speaker_id', (4,))], task_labels=[0]
Source loader sanity OK.


## 9. Custom Sweep Loop

Mirrors `phase_vib_stagger`'s pattern but threads `model_kwargs` through the variant trainer factory so each variant can set `vib_disabled_groups` / `vib_betas` independently. Per-variant config overrides (e.g., backbone, vib_beta) are also supported.


In [11]:
MODEL_CLASSES = {'GLADAModel': GLADAModel, 'PerGroupVIBModel': PerGroupVIBModel}

all_results = {}
sweep_start = time.time()
results_path = os.path.join(DRIVE_LOG_DIR, RESULTS_FILENAME)

for i, (name, spec) in enumerate(VARIANTS.items()):
    elapsed = (time.time() - sweep_start) / 60
    print(f'\n{"="*60}\nVariant {i+1}/{len(VARIANTS)}: {name}\n  elapsed: {elapsed:.1f} min\n{"="*60}')

    backbone = spec.get('backbone', None)
    cfg_kwargs = dict()
    if backbone is not None:
        cfg_kwargs['backbone'] = backbone
    cfg = make_config(name, **cfg_kwargs)
    for attr, val in spec.get('config_overrides', {}).items():
        setattr(cfg, attr, val)
    # If batch_size or effective_batch_size was overridden, re-derive
    # grad_accum_steps (matches GLADAConfig.__post_init__ formula).
    _bs_overrides = {'batch_size', 'effective_batch_size'} & set(spec.get('config_overrides', {}))
    if _bs_overrides:
        cfg.grad_accum_steps = max(1, cfg.effective_batch_size // (cfg.batch_size * 2))
        print(f'  [config] batch_size={cfg.batch_size}, effective_batch_size={cfg.effective_batch_size}, grad_accum_steps={cfg.grad_accum_steps}')
    set_seed(cfg.seed)

    # If this variant changes RawBoost, rebuild source_ds. Otherwise reuse.
    use_rb = spec.get('use_rawboost', RAWBOOST_DEFAULT)
    if use_rb != RAWBOOST_DEFAULT:
        from glada.augmentation import RawBoost as _RB
        aug = _RB if use_rb else None
        local_source_ds = ASVspoofDatasetWithSpeaker(
            protocol_path=ASV_PROTO, audio_dir=ASV_AUDIO,
            config=cfg, augmenter=aug,
        )
    else:
        local_source_ds = source_ds

    loaders = create_dataloaders(cfg, local_source_ds, target_ds, eval_ds, extra_eval)

    ModelCls = MODEL_CLASSES[spec['model_class']]
    model_kwargs = spec.get('model_kwargs', {})
    TrainerCls = GLADATrainer  # No staggered variants in Phase A/B

    class _VariantTrainer(TrainerCls):
        def __init__(self, *a, **kw):
            super().__init__(*a, **kw)
            old_model = self.model
            new_model = ModelCls(self.config, **model_kwargs).to(self.device)
            old_state = old_model.state_dict()
            new_state = new_model.state_dict()
            for k in new_state.keys():
                if k in old_state and old_state[k].shape == new_state[k].shape:
                    new_state[k] = old_state[k]
            new_model.load_state_dict(new_state, strict=False)
            self.model = new_model
            self._setup_phase1()
            trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in self.model.parameters())
            print(f'  [VARIANT] {ModelCls.__name__} kwargs={model_kwargs}; '
                  f'trainable={trainable:,}/{total:,}')

    try:
        trainer_kwargs = dict(
            config=cfg,
            source_loader=loaders['source'],
            target_loader=loaders['target'],
            eval_loader=loaders['eval'],
            extra_eval_loaders={k: v for k, v in loaders.items() if k not in ('source', 'target', 'eval')},
            num_speakers=num_speakers,
        )
        trainer = _VariantTrainer(**trainer_kwargs)
        results = trainer.train()
        all_results[name] = results
    except Exception as e:
        import traceback; traceback.print_exc()
        all_results[name] = {'error': str(e)}

    save = {k: {kk: vv for kk, vv in v.items() if kk != 'history'} for k, v in all_results.items()}
    with open(results_path, 'w') as f:
        json.dump(save, f, indent=2, default=str)

    if 'trainer' in dir(): del trainer
    torch.cuda.empty_cache(); gc.collect()

print(f'\nSweep complete: {len(all_results)} variants in {(time.time()-sweep_start)/60:.1f} min')



Variant 1/5: pergroup_vib_s123
  elapsed: 0.0 min
  Updating num_speakers: 107 -> 20

Building GLADA model: wavlm-base-plus


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

  Trainable: 6,356,140 / 100,738,076
  Run dir: /content/drive/MyDrive/Jakob/Experiments/phase_pgvib_replicate/pergroup_vib_s123/20260507_192942


... (321 outputs left out)

Ep 17:   0%|          | 0/1057 [00:00<?, ?it/s]

  Ep 17 P2: L=0.375 KL=0.22 | ITW=38.36% | 2019_LA=0.99%


Ep 18:   0%|          | 0/1057 [00:00<?, ?it/s]

  Ep 18 P2: L=0.376 KL=0.20 | ITW=39.42% | 2019_LA=0.84%
  Early stopping at epoch 18

Done in 227.2 min | Best ITW EER: 18.40% (Ep 3)

Sweep complete: 5 variants in 897.1 min


## 10. Decision Table

The four runs in this notebook combined with the existing seed-42 results give 3-seed measurements for both per_group_vib and drop_vib_g0. The headline comparison is per-group mean vs drop-g0 mean — the LOO finding requires this gap to persist across seeds.


In [12]:
REFERENCE = {
    'per_group_vib (BP, s42)':   11.67,  # existing seed-42 per-group VIB
    'drop_vib_g0 (BP, s42)':     17.61,  # existing seed-42 drop-g0
    'drop_vib_g1 (BP, s42)':     13.80,
    'drop_vib_g2 (BP, s42)':     11.99,
}

header = f'{"variant":<32s} {"ITW EER":>10s}' + ''.join(f'  vs {k:<18s}' for k in REFERENCE.keys())
print(header); print('-' * len(header))
for name, res in all_results.items():
    if 'error' in res:
        print(f'{name:<32s} FAILED: {res["error"][:60]}'); continue
    eer = res.get('best_eer') or res.get('eval_metrics', {}).get('ITW', {}).get('eer')
    if eer is None:
        print(f'{name:<32s} NO EER'); continue
    deltas = ''.join(f'  {(eer - v):>+18.2f}' for v in REFERENCE.values())
    print(f'{name:<32s} {eer:>10.2f}{deltas}')

print()
print('Reference (no retraining):')
for name, eer in REFERENCE.items():
    print(f'  {name:<32s} {eer:>10.2f}')


variant                             ITW EER  vs per_group_vib (BP, s42)  vs drop_vib_g0 (BP, s42)  vs drop_vib_g1 (BP, s42)  vs drop_vib_g2 (BP, s42)
-----------------------------------------------------------------------------------------------------------------------------------------------------
pergroup_vib_s123                     13.40               +1.73               -4.21               -0.40               +1.41
pergroup_vib_s456                     10.83               -0.84               -6.78               -2.97               -1.16
drop_vib_g0_s123                      14.46               +2.79               -3.15               +0.66               +2.47
drop_vib_g0_s456                      14.61               +2.94               -3.00               +0.81               +2.62
g0_only_xlsr_s42                      18.40               +6.73               +0.79               +4.60               +6.41

Reference (no retraining):
  per_group_vib (BP, s42)               11.67
  drop

## 11. Verdict & Multi-seed Summary

Two summaries: Phase 1 (3-seed mean ± std for per-group VIB and drop-g0, combining the existing seed-42 numbers with the two new seeds), and Phase 2 (xlsr-300m g0-only cross-backbone result).


In [13]:
import statistics
valid = {n: r for n, r in all_results.items() if 'error' not in r and r.get('best_eer') is not None}
if not valid:
    print('All variants failed; cannot summarize.'); raise SystemExit(0)

# Existing seed-42 results (from phase_a_leaveout)
S42_PERGROUP = 11.67
S42_DROP_G0  = 17.61

# === Phase 1: multi-seed replication summary ===
pergroup_eers = [S42_PERGROUP] + [valid[n]['best_eer'] for n in ('pergroup_vib_s123', 'pergroup_vib_s456') if n in valid]
drop_g0_eers  = [S42_DROP_G0]  + [valid[n]['best_eer'] for n in ('drop_vib_g0_s123', 'drop_vib_g0_s456') if n in valid]

def stats(name, eers):
    if len(eers) < 2:
        print(f'  {name}: insufficient seeds, have {eers}'); return None
    m, s = statistics.mean(eers), statistics.stdev(eers)
    print(f'  {name}: mean={m:.2f}%, std={s:.2f}% (n={len(eers)}, vals={[f"{v:.2f}" for v in eers]})')
    return m

print('=' * 70)
print('PHASE 1 — Multi-seed replication (combining existing s42 + this notebook):')
m_pg = stats('per_group_vib', pergroup_eers)
m_g0 = stats('drop_vib_g0   ', drop_g0_eers)
if m_pg is not None and m_g0 is not None:
    print(f'  Gap (drop-g0 mean - per-group mean): {m_g0 - m_pg:+.2f} pp')
    print('  -> If gap is positive and >> seed-noise, LOO finding is robust.')
print('=' * 70)

# === Phase 2: xlsr-300m cross-backbone ===
phase2 = {n: r for n, r in valid.items() if n.startswith('g0_only_xlsr_')}
if phase2:
    print()
    print('=' * 70)
    print('PHASE 2 — Cross-backbone (xlsr-300m g0-only, seed 42, batch_size=24):')
    for name, r in phase2.items():
        print(f'  {name}: {r["best_eer"]:.2f}% ITW EER')
    print('  Different SSL family (wav2vec2 lineage, multilingual) vs WavLM.')
    print('  If g0-only on xlsr also helps, finding generalizes across SSL families.')
    print('=' * 70)


PHASE 1 — Multi-seed replication (combining existing s42 + this notebook):
  per_group_vib: mean=11.97%, std=1.31% (n=3, vals=['11.67', '13.40', '10.83'])
  drop_vib_g0   : mean=15.56%, std=1.78% (n=3, vals=['17.61', '14.46', '14.61'])
  Gap (drop-g0 mean - per-group mean): +3.60 pp
  -> If gap is positive and >> seed-noise, LOO finding is robust.

PHASE 2 — Cross-backbone (xlsr-300m g0-only, seed 42, batch_size=24):
  g0_only_xlsr_s42: 18.40% ITW EER
  Different SSL family (wav2vec2 lineage, multilingual) vs WavLM.
  If g0-only on xlsr also helps, finding generalizes across SSL families.
